# Benchmark 03: Quantization — FP16 vs INT8 vs INT4

**Goal:** Measure the real impact of quantization on memory usage and inference speed.

**Setup:** Google Colab T4 GPU, HuggingFace Transformers (no vLLM needed)

**What we're measuring:**
- GPU memory used at each precision
- Time to first response
- Quality of output (does it degrade?)

**Why this matters:** Quantization is the most practical inference optimization — it directly attacks the memory-bandwidth bottleneck from the roofline model. Smaller weights = less data to move = faster inference.

In [ ]:
# Install dependencies — works on any Colab session, no version conflicts
!pip install transformers accelerate bitsandbytes -q

In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "facebook/opt-125m"
PROMPT = "Explain what a KV cache is in two sentences."
MAX_NEW_TOKENS = 60

def get_gpu_memory_mb():
    """Returns current GPU memory used in MB."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    return 0

def run_benchmark(model, tokenizer, label):
    """Run inference and measure time + memory."""
    torch.cuda.reset_peak_memory_stats()
    inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")
    
    # Warmup
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=10)
    
    # Timed run
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    torch.cuda.synchronize()
    elapsed = time.time() - start
    
    peak_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\n{'='*50}")
    print(f"Precision: {label}")
    print(f"Time:      {elapsed:.2f}s")
    print(f"Peak GPU:  {peak_mem:.0f} MB")
    print(f"Output:    {response[:150]}")
    
    return {"label": label, "time": elapsed, "memory_mb": peak_mem, "output": response[:150]}

print("Setup complete. Ready to benchmark.")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024 / 1024:.0f} MB")

## Experiment 1: FP16 baseline
Full precision — this is the standard. All weights stored as 16-bit floats.

In [ ]:
print("Loading FP16 model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to("cuda")

result_fp16 = run_benchmark(model_fp16, tokenizer, "FP16")

# Free memory before next experiment
del model_fp16
torch.cuda.empty_cache()
print("\nFP16 done. Memory cleared.")

## Experiment 2: INT8 quantization
Weights stored as 8-bit integers. Half the memory of FP16.
Uses bitsandbytes LLM.int8() — activations stay in FP16, only weights quantized.

In [ ]:
print("Loading INT8 model...")
quantization_config_int8 = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quantization_config_int8,
    device_map="auto"
)

result_int8 = run_benchmark(model_int8, tokenizer, "INT8")

del model_int8
torch.cuda.empty_cache()
print("\nINT8 done. Memory cleared.")

## Experiment 3: INT4 quantization (NF4)
Weights stored as 4-bit. Quarter the memory of FP16.
Uses bitsandbytes NF4 (NormalFloat4) — the quantization format used in QLoRA.

In [ ]:
print("Loading INT4 (NF4) model...")
quantization_config_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
model_int4 = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quantization_config_int4,
    device_map="auto"
)

result_int4 = run_benchmark(model_int4, tokenizer, "INT4 (NF4)")

del model_int4
torch.cuda.empty_cache()
print("\nINT4 done. Memory cleared.")

## Results Summary

In [ ]:
results = [result_fp16, result_int8, result_int4]

print("\n" + "="*65)
print("QUANTIZATION BENCHMARK — RESULTS SUMMARY")
print("="*65)
print(f"Model:  {MODEL}")
print(f"GPU:    {torch.cuda.get_device_name(0)}")
print(f"Prompt: '{PROMPT}'")
print()
print(f"{'Precision':<14} {'Time':<10} {'Peak GPU':<12} {'vs FP16 memory':<16} {'vs FP16 speed'}")
print("-"*65)

baseline_mem = results[0]['memory_mb']
baseline_time = results[0]['time']

for r in results:
    mem_ratio = f"{r['memory_mb']/baseline_mem:.2f}x"
    time_ratio = f"{r['time']/baseline_time:.2f}x"
    print(f"{r['label']:<14} {r['time']:<10.2f} {r['memory_mb']:<12.0f} {mem_ratio:<16} {time_ratio}")

print()
print("KEY INSIGHT:")
mem_reduction = (1 - results[2]['memory_mb']/baseline_mem) * 100
print(f"INT4 uses {mem_reduction:.0f}% less GPU memory than FP16.")
print(f"This directly reduces memory bandwidth pressure — the core bottleneck")
print(f"from the roofline model. Less data to move = faster token generation.")
print()
print("OUTPUT COMPARISON (quality check):")
for r in results:
    print(f"  {r['label']}: {r['output'][:100]}")